In [ ]:
# Initialization and environment setup
%run ../../init.py
import math
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from ipywidgets import interact, fixed
import ipywidgets as widgets
import helpers as hp

sns.set_theme(style="whitegrid", palette="deep")

# Deep Neural Networks
## Architecture, Representations, and Universal Approximation

This lecture explores deep feedforward architectures:
- **Limitations of Fixed Basis Functions**: The curse of dimensionality and the data manifold hypothesis
- **Multilayer Perceptrons (MLPs)**: Mathematical formulation of hierarchical layers
- **Universal Approximation**: The theorems of Cybenko (1989) and Hornik (1991)
- **Activation Functions**: Properties of $\mathrm{ReLU}$, Leaky $\mathrm{ReLU}$, $\mathrm{GELU}$, $\mathrm{SiLU}$, $\tanh$, and sigmoid
- **Weight-Space Symmetries**: Non-identifiability and permutation equivalence
- **Mixture Density Networks (MDNs)**: Handling multi-valued inverse problems
- **Empirical Evaluation**: Modular PyTorch MLPs on FashionMNIST

# Limitations of Fixed Basis Functions
## The Curse of Dimensionality

Single-layer linear basis models express predictions as $y(\mathbf{x}) = \mathbf{w}^\mathrm{T}\boldsymbol{\phi}(\mathbf{x})$ using fixed feature mappings $\boldsymbol{\phi}(\mathbf{x})$.

In high dimensions ($D \gg 1$), fixed basis models suffer fundamental limitations:
1. **Exponential Basis Growth**: Covering a $D$-dimensional unit hypercube with fixed local basis functions requires $M = K^D$ units, which is completely intractable.
2. **Volume Concentration**: In high dimensions, the volume of a hypersphere concentrates almost entirely in a thin shell near its surface, leaving the interior space empty.
3. **Data Manifold Hypothesis**: Real-world high-dimensional data (e.g. natural images) typically concentrates near low-dimensional non-linear manifolds embedded in $\mathbb{R}^D$.

> 💡 **Core Insight:** Rather than hand-crafting fixed basis functions $\boldsymbol{\phi}(\mathbf{x})$, deep neural networks **learn data-dependent basis functions directly from data**.

## Multilayer Networks: Mathematical Formulation

A feedforward neural network computes a sequence of parametric non-linear transformations across $L$ layers:

For input vector $\mathbf{z}^{(0)} = \mathbf{x} \in \mathbb{R}^D$, layer $l \in \{1, \dots, L\}$ computes:
$$\mathbf{a}^{(l)} = \mathbf{W}^{(l)} \mathbf{z}^{(l-1)} + \mathbf{b}^{(l)}$$
$$\mathbf{z}^{(l)} = h^{(l)}\left(\mathbf{a}^{(l)}\right)$$

where:
- $\mathbf{W}^{(l)} \in \mathbb{R}^{M_l \times M_{l-1}}$ is the weight parameter matrix
- $\mathbf{b}^{(l)} \in \mathbb{R}^{M_l}$ is the bias vector
- $\mathbf{a}^{(l)}$ is the vector of linear activations
- $h^{(l)}(\cdot)$ is an element-wise non-linear **activation function**
- $\mathbf{z}^{(l)}$ represents the hidden representation at layer $l$

For the final layer $L$, the activation function is matched to the distribution of targets $t$ (identity for regression, sigmoid for binary classification, softmax for multi-class).

## The Universal Approximation Theorem

Can a feedforward neural network approximate any continuous function?

> 📘 **Universal Approximation Theorem (Cybenko 1989; Hornik 1991):**
> Let $h(\cdot)$ be any continuous non-constant, bounded, and monotonically increasing activation function. For any continuous function $f: [0, 1]^D \to \mathbb{R}$ and any arbitrary precision $\epsilon > 0$, there exists a **single hidden layer network** with a finite number of neurons $M$ such that:

$$\sup_{\mathbf{x} \in [0, 1]^D} \left| \sum_{j=1}^M w_j h\left(\mathbf{v}_j^\mathrm{T} \mathbf{x} + b_j\right) - f(\mathbf{x}) \right| < \epsilon$$

<small>📖 *Source: [Cybenko, G. (1989). Approximation by superpositions of a sigmoidal function. Mathematics of Control, Signals and Systems, 2(4), 303–314](https://doi.org/10.1007/bf02551274)*</small>

> ⚠️ **The Depth Caveat:** While a single hidden layer is theoretically universal, the required width $M$ can be **exponentially large**. Deep networks achieve the same representational capacity with exponentially fewer parameters by reusing hierarchical features.

# Activation Functions in Deep Networks
## Non-Linear Transformations

Without non-linear activation functions, composing multiple linear layers $\mathbf{W}^{(2)}\mathbf{W}^{(1)}\mathbf{x}$ collapses to a single linear map $\mathbf{W}^\prime \mathbf{x}$.

Modern deep architectures employ distinct families of activations:

1. **Rectified Linear Unit (ReLU)**:
   $$h(a) = \max(0, a)$$
   Piecewise linear; avoids vanishing gradients for positive inputs; highly efficient.

2. **Leaky ReLU**:
   $$h(a) = \max(\alpha a, a), \quad \alpha \approx 0.01$$
   Prevents "dead neurons" by preserving a small gradient for negative inputs.

## Smooth & Probabilistic Activations

3. **Gaussian Error Linear Unit (GELU)**:
   $$h(a) = a \cdot \Phi(a) = a \cdot P(X \le a), \quad X \sim \mathcal{N}(0, 1)$$
   Smooth non-linearity combining deterministic gating with stochastic regularisation; standard in modern Transformers.

4. **Sigmoid $\sigma(a)$ and Hyperbolic Tangent $\tanh(a)$**:
   $$h(a) = \tanh(a) = \frac{e^a - e^{-a}}{e^a + e^{-a}}$$
   Smooth saturating functions; suffer from vanishing gradients in deep architectures ($h'(a) \to 0$ as $|a| \to \infty$).

In [ ]:
# Plotting helper for activation functions and their first derivatives
def plot_activation_curves(act_name='relu'):
    a = torch.linspace(-4, 4, 300, requires_grad=True)
    
    act_fns = {
        'relu': torch.relu,
        'leaky_relu': lambda x: torch.nn.functional.leaky_relu(x, negative_slope=0.1),
        'gelu': torch.nn.functional.gelu,
        'silu': torch.nn.functional.silu,
        'tanh': torch.tanh,
        'sigmoid': torch.sigmoid
    }
    
    fn = act_fns[act_name]
    y = fn(a)
    y.backward(torch.ones_like(a))
    grad = a.grad.detach().numpy()
    
    plt.figure(figsize=(7, 3.8))
    plt.plot(a.detach().numpy(), y.detach().numpy(), lw=2.5, color='royalblue', label=f'{act_name}(a)')
    plt.plot(a.detach().numpy(), grad, lw=2.0, linestyle='--', color='firebrick', label="d/da")
    plt.axhline(0, color='gray', lw=0.8, alpha=0.6)
    plt.axvline(0, color='gray', lw=0.8, alpha=0.6)
    plt.title(f"Activation Function: {act_name.upper()} & Derivative", fontsize=12)
    plt.xlabel("Activation input (a)")
    plt.ylabel("Output h(a)")
    plt.legend(loc='upper left')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
# Interactive exploration of activation functions and derivative saturation
interact(plot_activation_curves, act_name=['relu', 'leaky_relu', 'gelu', 'silu', 'tanh', 'sigmoid']);

## Weight-Space Symmetries
## Non-Identifiable Representations

In linear models, error surfaces under concave objectives are strictly unimodal. In deep neural networks, the mapping from weights to input-output functions is **many-to-one**.

Consider a hidden layer with $M$ identical neurons with odd activation functions (e.g. $\tanh(-a) = -\tanh(a)$):
1. **Permutation Symmetry**: We can permute the order of the $M$ hidden units in $M!$ ways without changing the network's output function.
2. **Sign-Flip Symmetry**: For activations with odd symmetry, flipping the signs of all incoming weights $\mathbf{w}_j$ and outgoing weight $w_{kj}$ simultaneously leaves the output identical ($2^M$ combinations).

Consequently, any single parameter vector $\mathbf{w}$ belongs to an equivalence class containing:
$$M! \cdot 2^M$$
distinct weight vectors that yield **identical network outputs** and identical error values.

> 💡 **Optimization Consequence:** The loss landscape of a deep network contains at least $M! \cdot 2^M$ equivalent global minima.

# Representation Learning & Hierarchies
## Distributed vs Local Representations

Deep networks perform **representation learning**: each layer constructs an increasingly abstract feature representation of the input.

```
Input x ---> Layer 1 (Low-level: Edges, Textures)
       ---> Layer 2 (Mid-level: Parts, Motifs)
       ---> Layer 3 (High-level: Object Classes, Concepts)
```

- **Local Representations (One-Hot / Clustering)**:
  Each concept is represented by a single dedicated unit. Representing $N$ disjoint concepts requires $N$ units.
- **Distributed Representations (Deep Features)**:
  Each concept is represented by a pattern of activity across many units, and each unit participates in representing many concepts.
  $M$ binary units can represent $2^M$ distinct configurations exponentially more efficiently!

## Canonical Error Functions in Deep Learning

The choice of network loss function derives directly from the maximum likelihood principle under assumed conditional distributions $p(t \mid \mathbf{x})$:

1. **Regression with Gaussian Noise**:
   $$p(t \mid \mathbf{x}) = \mathcal{N}\left(t \mid y(\mathbf{x}), \sigma^2\right) \implies E(\mathbf{w}) = \frac{1}{2} \sum_{n=1}^N \left(t_n - y(\mathbf{x}_n)\right)^2$$

2. **Binary Classification with Bernoulli Likelihood**:
   $$p(t \mid \mathbf{x}) = y^t (1 - y)^{1 - t} \implies E(\mathbf{w}) = -\sum_{n=1}^N \left[t_n \ln y_n + (1 - t_n) \ln(1 - y_n)\right]$$

3. **Multi-Class Classification with Multinomial Likelihood**:
   $$p(\mathbf{t} \mid \mathbf{x}) = \prod_{k=1}^K y_k^{t_k} \implies E(\mathbf{w}) = -\sum_{n=1}^N \sum_{k=1}^K t_{nk} \ln y_{nk}$$
   paired with the softmax output activation $y_k = \frac{\exp(a_k)}{\sum_j \exp(a_j)}$.

# Mixture Density Networks (MDNs)
## Modeling Multimodal Conditional Distributions

Standard regression networks output the conditional mean $\mathbb{E}[t \mid \mathbf{x}]$.
When modeling **inverse problems**, the true forward mapping is one-to-many: multiple target values $t$ correspond to the exact same input $\mathbf{x}$ (multimodal conditional distribution).

A standard network trained on squared loss predicts the average across modes, generating unphysical outputs located in regions where the true target probability is zero!

**Mixture Density Networks (MDNs)** resolve this by predicting the parameters of a Gaussian Mixture Model (GMM) directly:
$$p(t \mid \mathbf{x}) = \sum_{k=1}^K \pi_k(\mathbf{x}) \mathcal{N}\left(t \mid \mu_k(\mathbf{x}), \sigma_k^2(\mathbf{x})\right)$$

where the network outputs:
1. **Mixing coefficients** $\pi_k(\mathbf{x}) \ge 0$ with $\sum \pi_k(\mathbf{x}) = 1$ (via softmax)
2. **Component means** $\mu_k(\mathbf{x})$ (linear outputs)
3. **Component variances** $\sigma_k^2(\mathbf{x}) > 0$ (via $\exp$ or softplus)

<small>📖 *Source: [Bishop, C. M. (1994). Mixture density networks. Technical Report NCRG/94/004, Aston University](https://doi.org/10.1007/978-3-031-45468-4)*</small>

In [ ]:
# Modular Multi-Layer Perceptron (MLP) Implementation in PyTorch
class ModularMLP(nn.Module):
    def __init__(self, in_features, hidden_dims, out_features, activation='relu'):
        super().__init__()
        layers = []
        dims = [in_features] + hidden_dims
        
        act_dict = {
            'relu': nn.ReLU,
            'tanh': nn.Tanh,
            'gelu': nn.GELU,
            'sigmoid': nn.Sigmoid
        }
        act_cls = act_dict.get(activation.lower(), nn.ReLU)
        
        for i in range(len(dims) - 1):
            layers.append(nn.Linear(dims[i], dims[i+1]))
            layers.append(act_cls())
            
        layers.append(nn.Linear(dims[-1], out_features))
        self.network = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.network(x)

# Instantiate a 3-layer deep MLP: 2 -> 64 -> 32 -> 1
mlp = ModularMLP(in_features=2, hidden_dims=[64, 32], out_features=1, activation='relu')
print(mlp)
total_params = sum(p.numel() for p in mlp.parameters())
print(f"Total Trainable Parameters: {total_params}")

In [ ]:
# Training and decision boundary helper for non-linear logic gates
def train_and_plot_gate(gate='xor', activation='relu', hidden_units=16):
    logic_data = {
        'and': (torch.tensor([[0., 0.], [1., 1.], [1., 0.], [0., 1.]]), torch.tensor([[0.], [1.], [0.], [0.]])),
        'or':  (torch.tensor([[0., 0.], [1., 1.], [1., 0.], [0., 1.]]), torch.tensor([[0.], [1.], [1.], [1.]])),
        'xor': (torch.tensor([[0., 0.], [1., 1.], [1., 0.], [0., 1.]]), torch.tensor([[0.], [0.], [1.], [1.]]))
    }
    X_gate, t_gate = logic_data[gate]
    
    torch.manual_seed(42)
    model = ModularMLP(in_features=2, hidden_dims=[hidden_units], out_features=1, activation=activation)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
    
    for _ in range(300):
        optimizer.zero_grad()
        loss = criterion(model(X_gate), t_gate)
        loss.backward()
        optimizer.step()
        
    plt.figure(figsize=(6, 4))
    xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 100), np.linspace(-0.5, 1.5, 100))
    grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
    with torch.no_grad():
        preds = torch.sigmoid(model(grid)).reshape(xx.shape).numpy()
        
    plt.contourf(xx, yy, preds, alpha=0.3, cmap='coolwarm')
    plt.scatter(X_gate[:, 0].numpy(), X_gate[:, 1].numpy(), c=t_gate.squeeze().numpy(), 
                cmap='coolwarm', s=120, edgecolors='k', lw=2)
    plt.title(f"MLP Decision Boundary: {gate.upper()} ({activation.upper()}, {hidden_units} hidden units)", fontsize=11)
    plt.xlabel("x_1"); plt.ylabel("x_2"); plt.tight_layout(); plt.show()

In [ ]:
# Interactive verification: Multi-Layer Perceptrons effortlessly solve XOR
interact(train_and_plot_gate, gate=['xor', 'and', 'or'], activation=['relu', 'tanh', 'gelu'], hidden_units=[4, 8, 16, 32]);

In [ ]:
# FashionMNIST Multiclass Classification Evaluation
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_dataset = datasets.FashionMNIST(root='../../shared/data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root='../../shared/data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

fash_model = ModularMLP(in_features=784, hidden_dims=[128, 64], out_features=10, activation='relu')
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(fash_model.parameters(), lr=0.003)

# Train single epoch demonstration
fash_model.train()
for batch_idx, (images, labels) in enumerate(train_loader):
    images = images.view(images.shape[0], -1)
    optimizer.zero_grad()
    loss = criterion(fash_model(images), labels)
    loss.backward()
    optimizer.step()
    if batch_idx >= 50: # Limit demonstration steps for fast interactive execution
        break

print(f"FashionMNIST Batch 50 Training Loss: {loss.item():.4f}")

In [ ]:
# Evaluate Model Accuracy on Test Set
fash_model.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        images = images.view(images.shape[0], -1)
        outputs = fash_model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        if total >= 1000:
            break

print(f"Test Accuracy on sample ({total} items): {100.0 * correct / total:.2f}%")

# Summary: Deep Neural Networks (Part 1)

In this lecture, we investigated the foundations of deep representation learning:

1. **Overcoming Fixed Basis Limits**:
   - Fixed basis mappings $\boldsymbol{\phi}(\mathbf{x})$ scale exponentially $K^D$ with dimension.
   - Deep networks learn adaptive, non-linear coordinate transformations aligned with low-dimensional data manifolds.

2. **Universal Approximation**:
   - Single-layer feedforward networks can approximate any continuous function arbitrarily well (Cybenko 1989).
   - However, depth provides exponential parameter efficiency over shallow architectures.

## Summary: Deep Neural Networks (Part 2)

3. **Activation Function Dynamics**:
   - Piecewise-linear activations ($\mathrm{ReLU}$, Leaky $\mathrm{ReLU}$, $\mathrm{GELU}$) avoid saturation and enable stable deep gradient propagation.

4. **Weight-Space Equivalence & Inverses**:
   - Network parameterizations contain $M! \cdot 2^M$ equivalent global minima due to permutation and sign symmetries.
   - Mixture Density Networks (MDNs) predict mixture parameters $\{\pi_k, \mu_k, \sigma_k^2\}$ to solve one-to-many inverse problems without averaging across modes.